# V-Modul 526 &mdash; Versuch 3
# Messen der Signale des MC4 Rezeptors: Luciferase-Reportergen-Assay

**Auswertung der Konzentrations-Wirkungs-Kurve (KWK) mit Python / JupyterLite**

---

### Worum geht es?

Der MC4 Rezeptor ist ein G<sub>s</sub>-gekoppelter GPCR. Nach Bindung von &alpha;-MSH aktiviert
G&alpha;<sub>s</sub> die Adenylylcyclase &rarr; cAMP &uarr; &rarr; PKA &rarr; CREB-P &rarr; Bindung an das
*cAMP response element* (CRE). Hinter dem CRE steht in diesem Versuch das Reportergen der
**Firefly-Luciferase**. Die gemessene Lumineszenz (Counts/s) ist damit ein Mass fuer die
G<sub>s</sub>-Signalaktivitaet des Rezeptors.

Verglichen werden drei Konditionen:

| Kondition | Bedeutung |
|---|---|
| **MC4R** | wildtypischer MC4 Rezeptor |
| **mut. MC4R** | MC4 Rezeptor mit der Adipositas-Mutation Ile194Phe |
| **pcDps** | Leervektor &ndash; Negativkontrolle |

### Was macht dieses Notebook?

1. Rohdaten (384-Well-Platte, Tecan Spark) direkt aus der Excel-Datei einlesen
2. Wells ueber das Pipettierschema (Skript S. 28, Abb. 7) den Konditionen zuordnen
3. Mittelwert und Standardabweichung der Triplikate berechnen
4. Normierung auf den **unstimulierten Leervektor** (pcDps, 0 M) &rarr; *x-fold of pcDps unstimulated*
5. Grafische Darstellung der KWK und der Kontrollen
6. Kurvenanpassung (*log(agonist) vs. response*, 3 Parameter) und Bestimmung der EC<sub>50</sub>-Werte

---
## 1. Module importieren

Alle benoetigten Pakete werden hier &ndash; und nur hier &ndash; geladen.

In **JupyterLite** laeuft Python im Browser (Pyodide). `numpy`, `pandas`, `scipy` und
`matplotlib` sind dort vorinstalliert und werden hier geladen.

Es wird **nichts nachinstalliert**: Die Excel-Datei wird in Abschnitt 3.1 allein mit
der Standardbibliothek gelesen. Damit gibt es keine Abhaengigkeit, die beim Start
fehlschlagen kann, und dieselbe Zelle funktioniert in JupyterLite wie in einem
lokalen Jupyter.

In [ ]:
# ==========================================================================
# Alle benoetigten Module. Bewusst ohne Nachinstallation und ohne "await":
# die Excel-Datei wird weiter unten nur mit der Standardbibliothek gelesen
# (eine .xlsx-Datei ist ein ZIP-Archiv mit XML-Dateien). Damit laeuft das
# Notebook in JupyterLite genauso wie in einem lokalen Jupyter.
# ==========================================================================

# --- Standardbibliothek ---------------------------------------------------
import sys
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

# --- Numerik und Daten ----------------------------------------------------
import numpy as np
import pandas as pd

# --- Kurvenanpassung ------------------------------------------------------
from scipy.optimize import curve_fit

# --- Grafik ---------------------------------------------------------------
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

# --- globale Darstellungseinstellungen ------------------------------------
mpl.rcParams.update({
    "figure.dpi":        110,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "font.family":       "sans-serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

IN_JUPYTERLITE = (sys.platform == "emscripten") or ("pyodide" in sys.modules)

print("Umgebung  :", "JupyterLite (Pyodide)" if IN_JUPYTERLITE else "lokales Jupyter")
print("Python    :", sys.version.split()[0])
print("numpy     :", np.__version__)
print("pandas    :", pd.__version__)
print("matplotlib:", mpl.__version__)
print("\nAlle Module geladen - openpyxl wird nicht benoetigt.")

---
## 2. Versuchsaufbau und Pipettierschema

### 2.1 Belegung der 384-Well-Platte (Skript S. 28, Abbildung 7)

Jede Praktikumsgruppe belegt einen Block aus **6 Spalten**. Innerhalb eines Blocks gilt:

* **Spalten 1&ndash;3 des Blocks** &rarr; MC4R (ungerade Zeilen) bzw. mut. MC4R (gerade Zeilen)
* **Spalten 4&ndash;6 des Blocks** &rarr; pcDps (nur ungerade Zeilen)

Die Zeilen codieren die Stimulation (Abb. 8: eine 96-Well-Zeile bedient je zwei 384-Well-Zeilen):

| Zeilen | Stimulation |
|---|---|
| A / B | 10 &micro;M Forskolin (Positivkontrolle) |
| C / D | 0 M &alpha;-MSH (DMEM, unstimuliert) |
| E / F | 10<sup>&minus;11</sup> M &alpha;-MSH |
| G / H | 10<sup>&minus;10</sup> M &alpha;-MSH |
| I / J | 10<sup>&minus;9</sup> M &alpha;-MSH |
| K / L | 10<sup>&minus;8</sup> M &alpha;-MSH |
| M / N | 10<sup>&minus;7</sup> M &alpha;-MSH |
| O / P | 10<sup>&minus;6</sup> M &alpha;-MSH |

Dabei gilt: **ungerade Zeile** (A, C, E, G, I, K, M, O) = MC4R bzw. pcDps,
**gerade Zeile** (B, D, F, H, J, L, N, P) = mut. MC4R.

### 2.2 Unser Block

Unsere Messwerte stehen in **Spalte 13 bis Spalte 18, Zeile A bis Zeile P**:

* Spalten **13, 14, 15** = Triplikate MC4R / mut. MC4R
* Spalten **16, 17, 18** = Triplikate pcDps

In [ ]:
# ============================================================================
# KONFIGURATION  --  hier und nur hier muss bei einer anderen Gruppe
#                    bzw. einer anderen Messdatei etwas geaendert werden.
# ============================================================================

# Dateiname der Rohdaten (Tecan-Spark-Export)
DATEINAME = "1_3_6_7_ONE_Glo_Lumineszenz_Modified_20260910_145312_1.xlsx"

# --- Unser Plattenblock: Spalten 13-18, Zeilen A-P -------------------------
SPALTEN_REZEPTOR = [13, 14, 15]   # Triplikate MC4R (ungerade Zeilen) / mut. MC4R (gerade Zeilen)
SPALTEN_PCDPS    = [16, 17, 18]   # Triplikate pcDps (nur ungerade Zeilen)

# --- Zeilenpaare -> Stimulation (Skript S. 28, Abb. 7 und Abb. 8) ----------
#   (Zeile MC4R/pcDps, Zeile mut. MC4R): Bedingung
ZEILEN_SCHEMA = [
    ("A", "B", "FSK"),
    ("C", "D", "DMEM"),
    ("E", "F", -11),
    ("G", "H", -10),
    ("I", "J",  -9),
    ("K", "L",  -8),
    ("M", "N",  -7),
    ("O", "P",  -6),
]

# --- Konstrukte -----------------------------------------------------------
KONSTRUKTE = ["MC4R", "mut. MC4R", "pcDps"]

# alle belegten Plattenzeilen, in der Reihenfolge des Schemas
ALLE_ZEILEN = [z for paar in ZEILEN_SCHEMA for z in paar[:2]]

# Reihenfolge der Bedingungen in den Ergebnistabellen
BEDINGUNGEN = ["FSK", "DMEM", -11, -10, -9, -8, -7, -6]

# Die alpha-MSH-Konzentrationen der KWK (ohne Kontrollen)
KONZENTRATIONEN = [-11, -10, -9, -8, -7, -6]

# Position, an der der unstimulierte Wert (DMEM) in der Prism-Darstellung
# auf der log-Achse aufgetragen wird (vgl. Auswertungsvorlage: "-13")
X_DMEM = -13.0

# Bezugsgroesse der Normierung: unstimulierter Leervektor
REFERENZ_KONSTRUKT = "pcDps"
REFERENZ_BEDINGUNG = "DMEM"

# --- Farben (angelehnt an die Auswertungsvorlage) -------------------------
FARBEN_EXCEL = {"MC4R": "#9CC3E5", "mut. MC4R": "#F0A860", "pcDps": "#4F9A3D"}
FARBEN_PRISM = {"MC4R": "#5B9BD5", "mut. MC4R": "#C1C641", "pcDps": "#C9959B"}
FARBE_FSK, FARBE_DMEM = "#1F6E8C", "#E8833A"

print(f"Block  : Spalten {SPALTEN_REZEPTOR + SPALTEN_PCDPS}, Zeilen A-P")
print(f"Normierung auf: {REFERENZ_KONSTRUKT} / {REFERENZ_BEDINGUNG} (unstimulierter Leervektor)")

---
## 3. Rohdaten einlesen

Der Tecan-Spark-Export enthaelt zuerst einen langen Geraete-Kopf (Methode, Firmware,
Messparameter &hellip;) und erst danach das eigentliche Plattenraster. Dieses beginnt mit
einer Zeile, deren erste Zelle `<>` enthaelt; darunter folgen die Zeilen `A` bis `P`.

Die folgende Funktion sucht diesen Marker selbststaendig &ndash; sie funktioniert also
auch, wenn sich die Laenge des Kopfes bei einem anderen Export aendert.

### Hinweis zum Oeffnen der Datei in JupyterLite

> Ziehen Sie die Excel-Datei per Drag &amp; Drop in den **Dateibrowser links** in
> JupyterLite (am besten in denselben Ordner wie dieses Notebook oder in einen
> Unterordner `daten/`). Die naechste Zelle findet sie dann automatisch.

### 3.1 Excel-Datei ohne Zusatzpakete lesen

Eine `.xlsx`-Datei ist ein ZIP-Archiv mit XML-Dateien. Sie laesst sich deshalb
allein mit der Standardbibliothek auslesen (`zipfile` und `xml.etree`).

Das ist hier bewusst so geloest: In JupyterLite muesste `openpyxl` sonst erst
ueber das Netz nachgeladen werden. Schlaegt das fehl oder ist die Verbindung
langsam, bricht die Importzelle ab &ndash; und danach fehlen alle weiteren
Module. Ohne diese Abhaengigkeit laeuft das Notebook in JupyterLite genauso
zuverlaessig wie lokal.

In [ ]:
# ==========================================================================
# Excel-Datei ohne Zusatzpakete lesen
# --------------------------------------------------------------------------
# Eine .xlsx-Datei ist ein ZIP-Archiv mit XML-Dateien. Sie laesst sich daher
# allein mit zipfile und xml.etree aus der Standardbibliothek auslesen. Das
# ist in JupyterLite zuverlaessiger als openpyxl, das dort erst ueber das
# Netz nachinstalliert werden muesste.
# ==========================================================================

XML_HAUPT = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
XML_BEZUG = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"


def _spaltenindex(bezug):
    """Wandelt einen Zellbezug wie 'N55' in den 0-basierten Spaltenindex (13)."""
    index = 0
    for zeichen in bezug or "":
        if zeichen.isalpha():
            index = index * 26 + (ord(zeichen.upper()) - 64)
        elif index:
            break
    return max(index - 1, 0)


def _zellwert(zelle, texte):
    """Liest den Wert einer Zelle; bei Formeln das zuletzt gespeicherte Ergebnis."""
    typ = zelle.get("t")

    if typ == "inlineStr":
        knoten = zelle.find(XML_HAUPT + "is")
        if knoten is None:
            return None
        return "".join(t.text or "" for t in knoten.iter(XML_HAUPT + "t"))

    knoten = zelle.find(XML_HAUPT + "v")
    if knoten is None or knoten.text is None:
        return None
    roh = knoten.text

    if typ == "s":                                   # Verweis auf sharedStrings
        nummer = int(roh)
        return texte[nummer] if 0 <= nummer < len(texte) else None
    if typ == "b":
        return roh == "1"
    if typ == "e":                                   # Fehlerwert wie #DIV/0!
        return None
    if typ == "str":
        return roh

    try:
        zahl = float(roh)
    except ValueError:
        return roh
    return int(zahl) if zahl.is_integer() else zahl


def lies_arbeitsmappe(pfad):
    """Liest alle Tabellenblaetter einer .xlsx-Datei.

    Rueckgabe: dict {Blattname: Liste von Zeilen}, jede Zeile eine Liste von
    Zellwerten. Leere Zeilen und Zellen bleiben erhalten, damit die
    angezeigten Zeilennummern denen in Excel entsprechen.
    """
    pfad = Path(pfad)
    if not zipfile.is_zipfile(pfad):
        raise ValueError(
            f"'{pfad.name}' ist keine .xlsx-Datei. Aeltere .xls-Dateien und "
            "CSV-Dateien werden nicht unterstuetzt - bitte in Excel oder "
            "LibreOffice als .xlsx speichern."
        )

    with zipfile.ZipFile(pfad) as archiv:
        vorhanden = set(archiv.namelist())

        # gemeinsam genutzte Zeichenketten
        texte = []
        if "xl/sharedStrings.xml" in vorhanden:
            for eintrag in ET.fromstring(archiv.read("xl/sharedStrings.xml")):
                texte.append("".join(t.text or "" for t in eintrag.iter(XML_HAUPT + "t")))

        # Zuordnung der Blattnamen zu den XML-Dateien
        ziele = {}
        if "xl/_rels/workbook.xml.rels" in vorhanden:
            for beziehung in ET.fromstring(archiv.read("xl/_rels/workbook.xml.rels")):
                ziele[beziehung.get("Id")] = beziehung.get("Target", "")

        blaetter = []
        for blatt in ET.fromstring(archiv.read("xl/workbook.xml")).iter(XML_HAUPT + "sheet"):
            datei = ziele.get(blatt.get(XML_BEZUG + "id"), "").split("/")[-1]
            kandidat = f"xl/worksheets/{datei}"
            if kandidat in vorhanden:
                blaetter.append((blatt.get("name") or f"Blatt{len(blaetter) + 1}", kandidat))

        if not blaetter:                              # Notfall: alles nehmen, was da ist
            blaetter = [(f"Blatt{n}", name) for n, name in
                        enumerate(sorted(x for x in vorhanden
                                         if x.startswith("xl/worksheets/") and x.endswith(".xml")), 1)]

        mappe = {}
        for name, datei in blaetter:
            zeilen = []
            for zeile in ET.fromstring(archiv.read(datei)).iter(XML_HAUPT + "row"):
                try:
                    nummer = int(zeile.get("r"))
                except (TypeError, ValueError):
                    nummer = len(zeilen) + 1
                while len(zeilen) < nummer - 1:       # uebersprungene Leerzeilen
                    zeilen.append([])

                werte = []
                for zelle in zeile.iter(XML_HAUPT + "c"):
                    spalte = _spaltenindex(zelle.get("r"))
                    while len(werte) < spalte:        # uebersprungene Leerzellen
                        werte.append(None)
                    werte.append(_zellwert(zelle, texte))
                zeilen.append(werte)
            mappe[name] = zeilen

    return mappe

In [ ]:
BUCHSTABEN = "ABCDEFGHIJKLMNOP"      # moegliche Zeilenbeschriftungen einer Platte
WELL_MUSTER = re.compile(r"^([A-P])\s*(\d{1,2})$")


def finde_messdateien(muster="*.xlsx"):
    """Listet alle Excel-Dateien an den in JupyterLite / Jupyter ueblichen Orten."""
    suchorte = [
        Path.cwd(), Path.cwd() / "daten", Path.cwd() / "data",
        Path.cwd().parent, Path.cwd().parent / "daten", Path.cwd().parent / "data",
        Path("/drive"), Path("/drive/daten"), Path("/drive/data"),   # JupyterLite-Laufwerk
    ]

    gefunden = []
    for ort in suchorte:
        try:
            if ort.is_dir():
                gefunden += [p for p in sorted(ort.glob(muster))
                             if not p.name.startswith("~$")]
        except OSError:
            continue

    eindeutig, bekannt = [], set()
    for pfad in gefunden:
        schluessel = str(pfad.resolve())
        if schluessel not in bekannt:
            bekannt.add(schluessel)
            eindeutig.append(pfad.resolve())
    return eindeutig


def _zahl(wert):
    """Wandelt einen Zellwert in float um; alles Unlesbare wird zu NaN.

    Faengt Exporte ab, in denen Zahlen als Text oder mit Komma als
    Dezimaltrennzeichen gespeichert sind.
    """
    if wert is None or isinstance(wert, bool):
        return np.nan
    if isinstance(wert, (int, float)):
        return float(wert)
    text = str(wert).strip().replace(" ", "")
    if not text:
        return np.nan
    try:
        return float(text)
    except ValueError:
        pass
    try:                                   # z.B. "1.234,5" oder "1234,5"
        return float(text.replace(".", "").replace(",", "."))
    except ValueError:
        return np.nan


def _text(zeile, spalte=0):
    """Zellinhalt als bereinigter Grossbuchstaben-Text, sonst leerer String."""
    if not zeile or spalte >= len(zeile):
        return ""
    wert = zeile[spalte]
    return wert.strip().upper() if isinstance(wert, str) else ""


def _transponiere(zellen):
    """Vertauscht Zeilen und Spalten (fuer transponiert exportierte Platten)."""
    breite = max((len(z) for z in zellen), default=0)
    return [[z[k] if k < len(z) else None for z in zellen] for k in range(breite)]


def _breite_aus_daten(daten):
    """Anzahl der Spalten bis zum letzten Messwert."""
    return max((k + 1 for werte in daten.values()
                for k, v in enumerate(werte) if np.isfinite(v)), default=0)


def _raster_aus_gitter(zellen, blattname, quelle):
    """Sucht Bloecke aufeinanderfolgender Zeilen A, B, C, ... mit Messwerten.

    Die Beschriftung darf in einer der ersten drei Spalten stehen; es wird
    bewusst NICHT nach der '<>'-Kopfzeile gesucht, da dieser Marker je nach
    Export auch an anderen Stellen der Datei vorkommt.
    """
    bloecke = []
    breite_max = max((len(z) for z in zellen), default=0)

    for spalte in range(min(3, max(breite_max, 1))):
        i = 0
        while i < len(zellen):
            if _text(zellen[i], spalte) != "A":       # ein Raster beginnt immer bei A
                i += 1
                continue

            daten, naechste, j = {}, 0, i
            while j < len(zellen) and naechste < len(BUCHSTABEN):
                if _text(zellen[j], spalte) != BUCHSTABEN[naechste]:
                    break
                werte = [_zahl(v) for v in zellen[j][spalte + 1:]]
                if not any(np.isfinite(w) for w in werte):     # Zeile ohne Messwerte
                    break
                daten[BUCHSTABEN[naechste]] = werte
                naechste += 1
                j += 1

            if len(daten) >= 4:                       # zu kurze Bloecke sind keine Platte
                breite = _breite_aus_daten(daten)
                if breite > 0:
                    bloecke.append({"blatt": blattname, "zeile": i + 1, "daten": daten,
                                    "breite": breite, "quelle": quelle})
                i = max(j, i + 1)
            else:
                i += 1

    return bloecke


def _ist_laufindex(werte):
    """Erkennt eine fortlaufende Nummerierung (1, 2, 3, ...) als Wertspalte."""
    if len(werte) < 3 or not all(float(w).is_integer() for w in werte):
        return False
    start = int(werte[0])
    return [int(w) for w in werte] == list(range(start, start + len(werte)))


def _raster_aus_liste(zellen, blattname):
    """Baut das Raster aus einem Listenexport (Spalte mit Wellnamen wie 'A1', 'B13')."""
    breite_max = max((len(z) for z in zellen), default=0)

    for spalte in range(breite_max):
        treffer = []
        for zeile in zellen:
            m = WELL_MUSTER.match(_text(zeile, spalte))
            if m:
                treffer.append((zeile, m.group(1), int(m.group(2))))
        if len(treffer) < 24:                          # zu wenige Wells fuer eine Platte
            continue

        # Wertspalte suchen: bevorzugt rechts der Wellspalte. Uebersprungen werden
        # fortlaufende Nummerierungen und konstante Spalten (z.B. Temperatur).
        reihenfolge = sorted((k for k in range(breite_max) if k != spalte),
                             key=lambda k: (0 if k > spalte else 1, k))
        beste = None
        for kandidat in reihenfolge:
            werte = [_zahl(zeile[kandidat] if kandidat < len(zeile) else None)
                     for zeile, _, _ in treffer]
            gueltig = [w for w in werte if np.isfinite(w)]
            if len(gueltig) < 24 or len(set(gueltig)) <= 1 or _ist_laufindex(gueltig):
                continue
            beste = kandidat
            break
        if beste is None:
            continue

        roh, breite = {}, 0
        for zeile, buchstabe, nummer in treffer:
            wert = _zahl(zeile[beste] if beste < len(zeile) else None)
            roh.setdefault(buchstabe, {})[nummer] = wert
            breite = max(breite, nummer)

        daten = {b: [roh[b].get(k + 1, np.nan) for k in range(breite)]
                 for b in BUCHSTABEN if b in roh}
        if len(daten) >= 4 and breite > 0:
            return [{"blatt": blattname, "zeile": 1, "daten": daten,
                     "breite": breite, "quelle": "Listenformat"}]

    return []


def _sammle_raster(zellen, blattname):
    """Wendet alle bekannten Layouts auf ein Tabellenblatt an."""
    bloecke  = _raster_aus_gitter(zellen, blattname, "Raster")
    bloecke += _raster_aus_gitter(_transponiere(zellen), blattname, "Raster (transponiert)")
    bloecke += _raster_aus_liste(zellen, blattname)

    # Mehrfachfunde desselben Blocks entfernen
    eindeutig, gesehen = [], set()
    for block in bloecke:
        kennung = (block["blatt"], block["quelle"], "".join(block["daten"].keys()),
                   block["breite"], tuple(block["daten"][k][0] for k in block["daten"]))
        if kennung not in gesehen:
            gesehen.add(kennung)
            eindeutig.append(block)
    return eindeutig


def zeige_dateistruktur(pfad, max_zeilen=30, max_spalten=10, breite=13):
    """Zeigt den Aufbau der Datei - hilft, wenn kein Raster erkannt wurde."""
    mappe = lies_arbeitsmappe(pfad)
    print(f"Aufbau von '{Path(pfad).name}':")
    for blattname, zellen in mappe.items():
        spalten = max((len(z) for z in zellen), default=0)
        print(f"\n  Blatt '{blattname}': {len(zellen)} Zeilen x {spalten} Spalten")
        gezeigt = 0
        for nr, zeile in enumerate(zellen, 1):
            felder = [("" if v is None else str(v))[:breite] for v in zeile[:max_spalten]]
            if not any(f.strip() for f in felder):
                continue
            print(f"    {nr:>4} | " + " | ".join(f"{f:<{breite}}" for f in felder).rstrip())
            gezeigt += 1
            if gezeigt >= max_zeilen:
                print(f"    ... (weitere Zeilen nicht angezeigt)")
                break


def lies_plattenraster(pfad, melde=True):
    """Liest das groesste Plattenraster aus dem Export.

    Unterstuetzt drei Layouts: normales Raster, transponiertes Raster und
    Listenformat mit Wellnamen. Rueckgabe: (raster, kandidaten).
    """
    mappe = lies_arbeitsmappe(pfad)

    kandidaten = []
    for blattname, zeilen in mappe.items():
        kandidaten += _sammle_raster(zeilen, blattname)

    if not kandidaten:
        if melde:
            print("In der Datei wurde kein Plattenraster erkannt.\n")
            zeige_dateistruktur(pfad)
        raise ValueError(
            f"Kein Plattenraster in '{Path(pfad).name}' gefunden "
            f"(Blaetter: {list(mappe)}). "
            "Der Aufbau der Datei ist oben abgedruckt. Erwartet wird entweder ein "
            "Block aufeinanderfolgender Zeilen A, B, C, ... mit den Messwerten "
            "daneben, oder eine Liste mit Wellnamen wie 'A1', 'B13' und einer "
            "Wertespalte. Pruefen Sie, ob wirklich der Tecan-Export eingelesen "
            "wurde (Pfad siehe oben)."
        )

    # Das groesste Raster gewinnt: erst Zeilenzahl, dann Anzahl der Messwerte.
    def guete(block):
        messwerte = sum(1 for werte in block["daten"].values()
                        for v in werte if np.isfinite(v))
        return (len(block["daten"]), messwerte)

    bester = max(kandidaten, key=guete)
    for block in kandidaten:
        block["gewaehlt"] = block is bester

    breite = bester["breite"]
    daten = {marke: (werte + [np.nan] * breite)[:breite]
             for marke, werte in bester["daten"].items()}

    # Die Spaltennummern werden selbst gesetzt (Position 1 = Well-Spalte 1) und
    # nicht aus einer Kopfzeile uebernommen: dort stehen sie je nach Export als
    # Text und waeren dann nicht ueber die Zahl 13 ansprechbar.
    raster = pd.DataFrame(list(daten.values()), index=list(daten.keys()))
    raster.columns = pd.Index(range(1, breite + 1), dtype="int64", name="Spalte")
    raster.index.name = "Zeile"
    return raster, kandidaten


def waehle_messdatei(spalten, zeilen, dateiname=DATEINAME):
    """Waehlt aus allen gefundenen Excel-Dateien die mit dem passenden Raster.

    Entscheidend ist der INHALT, nicht der Dateiname: gesucht wird eine Datei,
    deren Plattenraster die konfigurierten Zeilen und Spalten enthaelt. So wird
    z.B. der 96-Well-Export des Oberflaechen-ELISA (Versuch 2) nicht
    versehentlich als Reportergen-Assay ausgewertet.

    Rueckgabe: (pfad, raster, kandidaten)
    """
    dateien = finde_messdateien()
    if not dateien:
        raise FileNotFoundError(
            "Es wurde keine Excel-Datei gefunden.\n\n"
            "-> Ziehen Sie den Lumineszenz-Export des Reportergen-Assays per "
            "Drag & Drop in den Dateibrowser links (gleicher Ordner wie dieses "
            "Notebook oder Unterordner 'daten/') und fuehren Sie diese Zelle "
            "erneut aus."
        )

    passend, bericht = [], []
    for pfad in dateien:
        try:
            raster, kandidaten = lies_plattenraster(pfad, melde=False)
        except (ValueError, KeyError, OSError, zipfile.BadZipFile) as fehler:
            # Datei enthaelt kein lesbares Plattenraster - naechste Datei pruefen.
            # Andere Fehler werden bewusst NICHT abgefangen, damit echte
            # Programmfehler sichtbar bleiben und nicht als "keine Daten" gelten.
            bericht.append((pfad, "kein Plattenraster erkannt", f"({type(fehler).__name__})"))
            continue

        fehlende = ([s for s in spalten if s not in raster.columns] +
                    [z for z in zeilen if z not in raster.index])
        form = f"Raster {raster.shape[0]} x {raster.shape[1]}"
        if fehlende:
            hinweis = ""
            if raster.shape[0] <= 8 and raster.shape[1] <= 12:
                hinweis = "(96-Well - vermutlich der ELISA aus Versuch 2)"
            bericht.append((pfad, f"{form} - Block nicht enthalten", hinweis))
        else:
            passend.append((pfad, raster, kandidaten))
            bericht.append((pfad, f"{form} - passt", ""))

    if len(dateien) > 1 or not passend:
        print(f"Gepruefte Excel-Dateien ({len(dateien)}):")
        for pfad, zustand, hinweis in bericht:
            marke = "   <== verwendet" if passend and pfad == passend[0][0] else ""
            print(f"  {pfad.name:<46} {zustand} {hinweis}{marke}")
        print()

    if not passend:
        raise FileNotFoundError(
            f"Keine der {len(dateien)} gefundenen Excel-Dateien enthaelt den "
            f"konfigurierten Block (Spalten {spalten[0]}-{spalten[-1]}, "
            f"Zeilen {zeilen[0]}-{zeilen[-1]}).\n\n"
            "Erwartet wird der Lumineszenz-Export des Reportergen-Assays: eine "
            "384-Well-Platte mit 16 Zeilen (A-P) und 24 Spalten.\n\n"
            "-> Ziehen Sie diese Datei in den Dateibrowser links. Der "
            "Oberflaechen-ELISA aus Versuch 2 (96-Well, 8 x 12) ist hier die "
            "falsche Datei.\n"
            "-> Mit zeige_dateistruktur(<pfad>) laesst sich der Aufbau einer "
            "einzelnen Datei ansehen."
        )

    # Bei mehreren passenden Dateien gewinnt der konfigurierte Dateiname.
    genau = [eintrag for eintrag in passend if eintrag[0].name == dateiname]
    return (genau or passend)[0]


def zeige_kandidaten(kandidaten):
    """Listet alle gefundenen Raster auf und markiert das verwendete."""
    print(f"Gefundene Raster in der Datei ({len(kandidaten)}):")
    for block in kandidaten:
        zeilen = "".join(block["daten"].keys())
        marke = "   <== verwendet" if block["gewaehlt"] else ""
        print(f"  Blatt '{block['blatt']}', ab Excel-Zeile {block['zeile']} "
              f"[{block['quelle']}]: Zeilen {zeilen} ({len(block['daten'])}), "
              f"{block['breite']} Spalten{marke}")


def pruefe_block(raster, spalten, zeilen, kandidaten=None):
    """Prueft, ob der konfigurierte Block wirklich im eingelesenen Raster liegt."""
    fehlende_spalten = [s for s in spalten if s not in raster.columns]
    fehlende_zeilen  = [z for z in zeilen  if z not in raster.index]
    if not (fehlende_spalten or fehlende_zeilen):
        return

    meldung = [
        "Der konfigurierte Block passt nicht zum eingelesenen Plattenraster.",
        f"  fehlende Spalten   : {fehlende_spalten}",
        f"  fehlende Zeilen    : {fehlende_zeilen}",
        f"  vorhandene Spalten : {list(raster.columns)}",
        f"  vorhandene Zeilen  : {list(raster.index)}",
    ]
    if kandidaten:
        meldung.append("  gefundene Raster in der Datei:")
        for block in kandidaten:
            zeilen_txt = "".join(block["daten"].keys())
            meldung.append(f"    Blatt '{block['blatt']}', ab Excel-Zeile "
                           f"{block['zeile']} [{block['quelle']}]: Zeilen "
                           f"{zeilen_txt}, {block['breite']} Spalten")
    meldung += [
        "",
        "Moegliche Ursachen:",
        "  - es wurde die falsche Excel-Datei eingelesen (Pfad siehe oben)",
        "  - der Export enthaelt eine 96-Well-Platte (Zeilen A-H, 12 Spalten)",
        "  - SPALTEN_REZEPTOR / SPALTEN_PCDPS passen nicht zum Block der Gruppe",
        "",
        "Mit zeige_dateistruktur(PFAD) laesst sich der Aufbau der Datei ansehen.",
    ]
    raise KeyError("\n".join(meldung))


# --- ausfuehren -----------------------------------------------------------
PFAD, platte, KANDIDATEN = waehle_messdatei(SPALTEN_REZEPTOR + SPALTEN_PCDPS, ALLE_ZEILEN)

print("Eingelesen:", PFAD, "\n")
zeige_kandidaten(KANDIDATEN)

print(f"\nPlattenformat: {platte.shape[0]} Zeilen x {platte.shape[1]} Spalten")
print(f"Zeilen : {list(platte.index)}")
print(f"Spalten: {list(platte.columns)}\n")

if platte.shape != (16, 24):
    print("ACHTUNG: erwartet wird eine 384-Well-Platte (16 Zeilen x 24 Spalten).\n")

platte

### 3.1 Fehlersuche: Aufbau der Datei ansehen

Diese Zelle ist **nur zur Fehlersuche** gedacht und muss im Normalfall nicht
ausgefuehrt werden. Sie zeigt den tatsaechlichen Aufbau der eingelesenen Datei
(alle Tabellenblaetter, die ersten Zeilen und Spalten) und hilft, wenn kein oder
ein falsches Raster erkannt wurde.

Das Einlesen beherrscht drei Layouts und waehlt automatisch das passende:

* **Raster** &ndash; Zeilen A&hellip;P untereinander, Messwerte daneben (Standard beim Tecan-Export)
* **Raster (transponiert)** &ndash; Buchstaben stehen in der Kopfzeile
* **Listenformat** &ndash; eine Spalte mit Wellnamen (`A1`, `B13`, &hellip;) und eine Wertespalte

In [ ]:
zeige_dateistruktur(PFAD, max_zeilen=25, max_spalten=10)

### 3.2 Kontrolle: unser Block

Zur Sicherheit wird nur der Ausschnitt angezeigt, der uns gehoert (Spalten 13&ndash;18,
Zeilen A&ndash;P). Die leeren Zellen in den geraden Zeilen (B, D, F, &hellip;) sind korrekt:
dort wurde laut Pipettierschema kein pcDps ausgesaet.

In [ ]:
pruefe_block(platte, SPALTEN_REZEPTOR + SPALTEN_PCDPS, ALLE_ZEILEN, KANDIDATEN)

unser_block = platte.loc[ALLE_ZEILEN, SPALTEN_REZEPTOR + SPALTEN_PCDPS]

print("Leere Zellen = laut Pipettierschema nicht belegt (pcDps nur in ungeraden Zeilen).\n")
unser_block

---
## 4. Wells den Konditionen zuordnen

Aus dem Plattenraster wird eine &bdquo;lange&ldquo; Tabelle (*tidy data*) erzeugt: eine Zeile
pro Well, mit Konstrukt, Bedingung, Well-Koordinate und Messwert. Das macht alle
folgenden Rechenschritte nachvollziehbar und leicht ueberpruefbar.

In [ ]:
def baue_messtabelle(raster):
    """Ordnet jedem Well des Blocks Konstrukt und Bedingung zu (Pipettierschema Abb. 7)."""
    eintraege = []

    for zeile_wt, zeile_mut, bedingung in ZEILEN_SCHEMA:

        # --- MC4R (wildtypisch): ungerade Zeile, Spalten 13-15 ---
        for spalte in SPALTEN_REZEPTOR:
            eintraege.append({
                "Konstrukt": "MC4R", "Bedingung": bedingung,
                "Well": f"{zeile_wt}{spalte}",
                "Lumineszenz": raster.at[zeile_wt, spalte],
            })

        # --- mut. MC4R: gerade Zeile, Spalten 13-15 ---
        for spalte in SPALTEN_REZEPTOR:
            eintraege.append({
                "Konstrukt": "mut. MC4R", "Bedingung": bedingung,
                "Well": f"{zeile_mut}{spalte}",
                "Lumineszenz": raster.at[zeile_mut, spalte],
            })

        # --- pcDps (Leervektor): ungerade Zeile, Spalten 16-18 ---
        for spalte in SPALTEN_PCDPS:
            eintraege.append({
                "Konstrukt": "pcDps", "Bedingung": bedingung,
                "Well": f"{zeile_wt}{spalte}",
                "Lumineszenz": raster.at[zeile_wt, spalte],
            })

    tabelle = pd.DataFrame(eintraege)
    tabelle["Konstrukt"] = pd.Categorical(tabelle["Konstrukt"], KONSTRUKTE, ordered=True)
    return tabelle


messwerte = baue_messtabelle(platte)

print(f"{len(messwerte)} Wells zugeordnet "
      f"({messwerte['Lumineszenz'].isna().sum()} davon ohne Messwert)\n")
messwerte.head(12)

### 4.1 Rohdaten als Uebersichtstabelle

Dieselben Daten noch einmal im Format der Auswertungsvorlage: pro Kondition und
Bedingung die drei Triplikate nebeneinander.

In [ ]:
triplikate = (messwerte
              .assign(Replikat=messwerte.groupby(["Konstrukt", "Bedingung"], observed=True).cumcount() + 1)
              .pivot_table(index=["Konstrukt", "Bedingung"], columns="Replikat",
                           values="Lumineszenz", observed=True, sort=False))
triplikate.columns = [f"Triplikat {i}" for i in triplikate.columns]
triplikate = triplikate.reindex(
    pd.MultiIndex.from_product([KONSTRUKTE, BEDINGUNGEN], names=["Konstrukt", "Bedingung"])
).dropna(how="all")

triplikate

---
## 5. Mittelwerte und Streuung der Triplikate

Fuer jede Kombination aus Konstrukt und Bedingung werden Mittelwert und
Standardabweichung der drei technischen Replikate berechnet.

> Die Standardabweichung beschreibt hier **nur die Pipettier-/Messstreuung innerhalb
> eines einzigen Experiments** (technische Replikate). Da der Versuch nur einmal
> durchgefuehrt wurde, wird sie laut Praktikumsskript in den Abbildungen **nicht** als
> Fehlerbalken dargestellt &ndash; sie dient ausschliesslich der Qualitaetsbeurteilung.

In [ ]:
kennwerte = (messwerte
             .groupby(["Konstrukt", "Bedingung"], observed=True)["Lumineszenz"]
             .agg(Mittelwert="mean",
                  SD=lambda s: s.std(ddof=1),     # Stichproben-SD, wie in Excel STABW.S
                  n="count")
             .reset_index())

# Variationskoeffizient als Mass fuer die Streuung der Triplikate
kennwerte["VK [%]"] = 100 * kennwerte["SD"] / kennwerte["Mittelwert"]

kennwerte = (kennwerte
             .set_index(["Konstrukt", "Bedingung"])
             .reindex(pd.MultiIndex.from_product([KONSTRUKTE, BEDINGUNGEN],
                                                 names=["Konstrukt", "Bedingung"])))
kennwerte

### 5.1 Auffaellige Triplikate

Ein einzelner stark abweichender Well verschiebt den Mittelwert &ndash; und damit, falls er
im Referenz-Well liegt, die gesamte Normierung. Die folgende Zelle listet deshalb alle
Triplikate mit einem Variationskoeffizienten &gt; 50 % auf und zeigt, wie sich der
Mittelwert ohne den am staerksten abweichenden Einzelwert veraendern wuerde.

> Die Werte werden **nicht automatisch entfernt**. Ob ein Ausreisser gestrichen wird,
> ist eine fachliche Entscheidung, die begruendet und im Protokoll dokumentiert werden
> muss.

In [ ]:
GRENZE_VK = 50.0   # Prozent

auffaellig = kennwerte[kennwerte["VK [%]"] > GRENZE_VK]

if auffaellig.empty:
    print(f"Keine Triplikate mit VK > {GRENZE_VK:.0f} % - Streuung unauffaellig.")
else:
    print(f"Triplikate mit VK > {GRENZE_VK:.0f} %:\n")
    for (konstrukt, bedingung) in auffaellig.index:
        gruppe = messwerte[(messwerte["Konstrukt"] == konstrukt) &
                           (messwerte["Bedingung"] == bedingung)]
        werte = gruppe["Lumineszenz"].to_numpy(float)
        wells = gruppe["Well"].tolist()

        # am staerksten vom Median abweichender Einzelwert
        idx_weg = int(np.argmax(np.abs(werte - np.median(werte))))
        ohne = np.delete(werte, idx_weg)

        print(f"  {konstrukt} / {bedingung}")
        paare = ", ".join(f"{w} = {int(v)}" for w, v in zip(wells, werte))
        print(f"     Wells        : {paare}")
        print(f"     Mittelwert   : {werte.mean():8.1f}  (VK {auffaellig.loc[(konstrukt, bedingung), 'VK [%]']:.0f} %)")
        print(f"     ohne {wells[idx_weg]:<5}   : {ohne.mean():8.1f}  "
              f"(= {ohne.mean() / werte.mean():.2f}-fach des urspruenglichen Werts)\n")

---
## 6. Normierung auf den unstimulierten Leervektor

Die absoluten Lumineszenz-Werte haengen von Zellzahl, Transfektionseffizienz und
Substratmenge ab und sind zwischen Platten nicht vergleichbar. Deshalb wird jeder
Mittelwert auf den Mittelwert des **unstimulierten Leervektors** (pcDps, 0 M &alpha;-MSH)
bezogen:

$$
\textsf{x-fold of pcDps unstimulated}
\;=\;
\frac{\overline{\textsf{RLU}}_{\textsf{Konstrukt, Bedingung}}}
     {\overline{\textsf{RLU}}_{\textsf{pcDps, 0 M}}}
$$

Damit ist pcDps / 0 M per Definition gleich 1. Alle Werte geben an, um welchen Faktor
das jeweilige Signal ueber (bzw. unter) der basalen Aktivitaet des Leervektors liegt.

In [ ]:
# --- Referenzwert: pcDps, unstimuliert -----------------------------------
referenz = kennwerte.loc[(REFERENZ_KONSTRUKT, REFERENZ_BEDINGUNG), "Mittelwert"]

print(f"Referenz (Mittelwert {REFERENZ_KONSTRUKT}, {REFERENZ_BEDINGUNG}): "
      f"{referenz:,.2f} Counts/s")

if not np.isfinite(referenz) or referenz <= 0:
    raise ValueError("Referenzwert ist 0 oder fehlt - Normierung nicht moeglich.")

# --- Normierung ----------------------------------------------------------
kennwerte["x-fold"]    = kennwerte["Mittelwert"] / referenz
kennwerte["SD x-fold"] = kennwerte["SD"]         / referenz

kennwerte[["Mittelwert", "SD", "n", "VK [%]", "x-fold", "SD x-fold"]]

### 6.1 Ergebnistabelle

Kompakte Darstellung der normierten Werte (Aufbau wie in der Auswertungsvorlage):
Zeilen = Konstrukte, Spalten = Stimulation.

In [ ]:
ergebnis = (kennwerte["x-fold"]
            .unstack("Bedingung")
            .reindex(index=KONSTRUKTE, columns=BEDINGUNGEN))
ergebnis.columns = ["FSK 10 µM", "DMEM (0 M)"] + [f"10^{c} M" for c in KONZENTRATIONEN]
ergebnis.index.name = "x-fold of pcDps unstimulated"

ergebnis.round(4)

---
## 7. Abbildung 1 &ndash; Konzentrations-Wirkungs-Kurve

Auftragung der normierten Signale gegen die &alpha;-MSH-Konzentration. Die x-Achse ist
kategorial (DMEM, dann 10<sup>&minus;11</sup> &hellip; 10<sup>&minus;6</sup> M), wie in der
Auswertungsvorlage. Ohne Fehlerbalken, da nur ein Experiment vorliegt.

In [ ]:
# --- Daten fuer die KWK (DMEM + alle alpha-MSH-Konzentrationen) -----------
kwk_bedingungen = ["DMEM"] + KONZENTRATIONEN
kwk = ergebnis.copy()
kwk.columns = BEDINGUNGEN
kwk = kwk[kwk_bedingungen]

x_kat  = np.arange(len(kwk_bedingungen))
labels = ["DMEM"] + [str(c) for c in KONZENTRATIONEN]

fig, ax = plt.subplots(figsize=(6.2, 4.0))

for konstrukt in KONSTRUKTE:
    ax.plot(x_kat, kwk.loc[konstrukt].values,
            marker="o", markersize=6, linewidth=2,
            color=FARBEN_EXCEL[konstrukt], label=konstrukt,
            markeredgecolor="white", markeredgewidth=0.6)

ax.set_title("reporter gene assay", fontsize=12, pad=12)
ax.set_xlabel(r"$\alpha$-MSH, FSK, log [M]", labelpad=8)
ax.set_ylabel("x-fold of pcDps unstimulated", labelpad=8)
ax.set_xticks(x_kat)
ax.set_xticklabels(labels)
ax.set_xlim(-0.4, len(x_kat) - 0.6)
ax.set_ylim(bottom=0)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18),
          ncol=3, frameon=False, handlelength=2.2)

fig.savefig("Abb1_Konzentrations-Wirkungs-Kurve.png")
plt.show()

---
## 8. Abbildung 2 &ndash; Kontrollen (Forskolin vs. DMEM)

Forskolin aktiviert die Adenylylcyclase **direkt**, also unabhaengig vom Rezeptor.
Die FSK-Kontrolle zeigt daher, ob Zellen, Transfektion, Reportergen und Substrat
ueberhaupt funktioniert haben. DMEM ist die zugehoerige unstimulierte Kontrolle.

In [ ]:
fsk  = [ergebnis.loc[k].iloc[0] for k in KONSTRUKTE]   # Spalte "FSK 10 µM"
dmem = [ergebnis.loc[k].iloc[1] for k in KONSTRUKTE]   # Spalte "DMEM (0 M)"

x     = np.arange(len(KONSTRUKTE))
breit = 0.35

fig, ax = plt.subplots(figsize=(5.6, 4.0))

ax.bar(x - breit / 2, fsk,  breit, label="FSK 10 µM", color=FARBE_FSK)
ax.bar(x + breit / 2, dmem, breit, label="DMEM",      color=FARBE_DMEM)

ax.axhline(1.0, color="#7F7F7F", linewidth=1.0, linestyle=":")

ax.set_title("FSK control", fontsize=12, pad=12)
ax.set_ylabel("x-fold of pcDps unstimulated", labelpad=8)
ax.set_xticks(x)
ax.set_xticklabels(KONSTRUKTE)
ax.set_ylim(bottom=0)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False)

fig.savefig("Abb2_FSK-Kontrolle.png")
plt.show()

---
## 9. Kurvenanpassung und EC<sub>50</sub>

Angepasst wird das in der Pharmakologie uebliche Modell
*log(agonist) vs. response* mit **drei Parametern** (variable Steigung fixiert auf 1,
Hill-Slope = 1):

$$
Y \;=\; \textsf{Bottom} \;+\; \frac{\textsf{Top} - \textsf{Bottom}}{1 + 10^{\,(\log EC_{50} \,-\, X)}}
$$

mit $X = \log_{10}(c_{\alpha\text{-MSH}} \,/\, \mathrm{M})$.

* **Bottom** &ndash; basales Signal ohne Agonist
* **Top** &ndash; maximales Signal bei saettigender Agonistkonzentration
* **EC<sub>50</sub>** &ndash; Konzentration, bei der das halbmaximale Signal erreicht wird
  (Mass fuer die Potenz des Agonisten am jeweiligen Rezeptor)

Wie in der Auswertungsvorlage geht der unstimulierte Wert (DMEM) als Punkt bei
$X = -13$ in den Fit ein; die Forskolin-Kontrolle bleibt aussen vor, da sie den
Rezeptor umgeht.

In [ ]:
def dosis_wirkung(x, bottom, top, log_ec50):
    """log(agonist) vs. response, 3 Parameter (Hill-Slope = 1)."""
    return bottom + (top - bottom) / (1.0 + 10.0 ** (log_ec50 - x))


def fitte_kurve(x, y):
    """Passt das 3-Parameter-Modell an und liefert Parameter + Guetemasse."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    gueltig = np.isfinite(x) & np.isfinite(y)
    x, y = x[gueltig], y[gueltig]

    ergebnis = {"Bottom": np.nan, "Top": np.nan, "LogEC50": np.nan, "EC50 [M]": np.nan,
                "Span": np.nan, "R²": np.nan, "n Punkte": len(x), "Status": ""}
    if len(x) < 4:
        ergebnis["Status"] = "zu wenige Punkte"
        return ergebnis

    start  = [float(np.min(y)), float(np.max(y)), float(np.median(x))]
    grenze = ([-np.inf, -np.inf, x.min() - 3], [np.inf, np.inf, x.max() + 3])

    try:
        popt, _ = curve_fit(dosis_wirkung, x, y, p0=start, bounds=grenze, maxfev=20000)
    except Exception as fehler:
        ergebnis["Status"] = f"Fit fehlgeschlagen ({type(fehler).__name__})"
        return ergebnis

    bottom, top, log_ec50 = popt
    rest  = y - dosis_wirkung(x, *popt)
    sq_r  = float(np.sum(rest ** 2))
    sq_t  = float(np.sum((y - y.mean()) ** 2))
    r2    = 1 - sq_r / sq_t if sq_t > 0 else np.nan

    ergebnis.update({
        "Bottom": bottom, "Top": top, "LogEC50": log_ec50,
        "EC50 [M]": 10.0 ** log_ec50, "Span": top - bottom, "R²": r2,
    })

    # Plausibilitaetspruefung: liegt die EC50 im gemessenen Bereich, gibt es ueberhaupt ein Fenster?
    hinweise = []
    if not (x.min() <= log_ec50 <= x.max()):
        hinweise.append("EC50 ausserhalb des gemessenen Bereichs")
    if abs(top - bottom) < 0.2 * max(abs(bottom), 1e-9):
        hinweise.append("kein nennenswertes Signalfenster (Top ≈ Bottom)")
    if np.isfinite(r2) and r2 < 0.8:
        hinweise.append("schlechte Anpassung (R² < 0,8)")
    ergebnis["Status"] = "; ".join(hinweise) if hinweise else "ok"
    return ergebnis


# --- x-Werte: DMEM bei -13, danach die echten Konzentrationen -------------
x_fit = np.array([X_DMEM] + [float(c) for c in KONZENTRATIONEN])

fit_ergebnisse = {}
for konstrukt in KONSTRUKTE:
    y = kennwerte.loc[konstrukt].loc[["DMEM"] + KONZENTRATIONEN, "x-fold"].to_numpy(float)
    fit_ergebnisse[konstrukt] = fitte_kurve(x_fit, y)

fits = pd.DataFrame(fit_ergebnisse).T
fits.index.name = "Nonlin fit"
fits

### 9.1 Ergebnisse der Kurvenanpassung

Darstellung im Format der Auswertungsvorlage (*Table of results*). Die Spalte
**Status** weist automatisch auf Fits hin, die rechnerisch zwar ein Ergebnis liefern,
inhaltlich aber nicht belastbar sind.

In [ ]:
fits_anzeige = pd.DataFrame({
    "Bottom":   fits["Bottom"].map(lambda v: f"{v:.4g}"),
    "Top":      fits["Top"].map(lambda v: f"{v:.4g}"),
    "Span":     fits["Span"].map(lambda v: f"{v:.4g}"),
    "LogEC50":  fits["LogEC50"].map(lambda v: f"{v:.4g}"),
    "EC50 [M]": fits["EC50 [M]"].map(lambda v: f"{v:.4g}" if np.isfinite(v) else "-"),
    "EC50 [nM]": fits["EC50 [M]"].map(lambda v: f"{v * 1e9:.4g}" if np.isfinite(v) else "-"),
    "R²":       fits["R²"].map(lambda v: f"{v:.4f}"),
    "n Punkte": fits["n Punkte"],
    "Status":   fits["Status"],
})
fits_anzeige.index.name = "Nonlin fit"
fits_anzeige

In [ ]:
# --- EC50 in gut lesbarer Form -------------------------------------------
print("EC50-Werte (log(agonist) vs. response, 3 Parameter)")
print("-" * 62)
for konstrukt in KONSTRUKTE:
    e = fit_ergebnisse[konstrukt]
    ec50 = e["EC50 [M]"]
    if np.isfinite(ec50):
        print(f"{konstrukt:<12}  EC50 = {ec50:.3e} M  = {ec50 * 1e9:,.3f} nM"
              f"   |  R² = {e['R²']:.4f}   |  {e['Status']}")
    else:
        print(f"{konstrukt:<12}  EC50 = nicht bestimmbar   |  {e['Status']}")
print("-" * 62)

verhaeltnis = fit_ergebnisse["mut. MC4R"]["EC50 [M]"] / fit_ergebnisse["MC4R"]["EC50 [M]"]
if np.isfinite(verhaeltnis):
    print(f"\nEC50(mut. MC4R) / EC50(MC4R) = {verhaeltnis:,.2f}"
          f"   ->  {'Rechtsverschiebung (geringere Potenz)' if verhaeltnis > 1 else 'Linksverschiebung (hoehere Potenz)'}"
          " der Mutante")

---
## 10. Abbildung 3 &ndash; Gesamtdarstellung (Auswertungsvorlage)

Zusammenfassende Abbildung im Stil der Vorlage: Messpunkte (Kreise) mit den
angepassten Kurven (gestrichelt), der unstimulierte Wert bei log[M] = &minus;13 und die
Forskolin-Positivkontrolle als Quadrate links neben der Konzentrationsachse
(&bdquo;pos ctrl&ldquo;).

In [ ]:
X_POSCTRL = -14.6          # Position der Positivkontrolle links der Achse

fig, ax = plt.subplots(figsize=(6.8, 4.8))

x_punkte = np.array([X_DMEM] + [float(c) for c in KONZENTRATIONEN])
x_glatt  = np.linspace(X_DMEM, KONZENTRATIONEN[-1], 400)

for konstrukt in KONSTRUKTE:
    farbe = FARBEN_PRISM[konstrukt]
    y = kennwerte.loc[konstrukt].loc[["DMEM"] + KONZENTRATIONEN, "x-fold"].to_numpy(float)

    # Messpunkte
    ax.plot(x_punkte, y, linestyle="none", marker="o", markersize=7,
            color=farbe, label=konstrukt, zorder=3)

    # angepasste Kurve
    e = fit_ergebnisse[konstrukt]
    if np.isfinite(e["LogEC50"]):
        ax.plot(x_glatt, dosis_wirkung(x_glatt, e["Bottom"], e["Top"], e["LogEC50"]),
                linestyle="--", linewidth=1.5, color=farbe, zorder=2)

    # Positivkontrolle (10 µM Forskolin) als Quadrat
    ax.plot(X_POSCTRL, kennwerte.loc[(konstrukt, "FSK"), "x-fold"],
            linestyle="none", marker="s", markersize=7, color=farbe, zorder=3)

# --- Achsen im Stil der Vorlage ------------------------------------------
ticks  = [X_POSCTRL] + list(range(-13, -5))
labels = ["pos\nctrl"] + [str(t) for t in range(-13, -5)]
ax.set_xticks(ticks)
ax.set_xticklabels(labels)
ax.set_xlim(X_POSCTRL - 0.8, -5.4)

ax.set_xlabel(r"$\alpha$-MSH, FSK, log [M]", fontweight="bold", labelpad=8)
ax.set_ylabel("x-fold of pcDps unstimulated", fontweight="bold", labelpad=8)
ax.set_title("Reporter gene assay G$_\\mathrm{s}$", fontweight="bold", pad=14)

obergrenze = float(np.nanmax(kennwerte["x-fold"].to_numpy(float)))
ax.set_ylim(0, obergrenze * 1.25)

# Achsen leicht abgesetzt (Prism-Optik)
ax.spines["left"].set_position(("outward", 8))
ax.spines["bottom"].set_position(("outward", 8))
ax.tick_params(direction="out", length=5, width=1.0)

# optische Trennung der Positivkontrolle von der Konzentrationsachse
ax.axvline(X_POSCTRL + 0.75, color="#BFBFBF", linewidth=0.8, linestyle=":")

griffe = [plt.Line2D([], [], linestyle="none", marker="o", markersize=7,
                     color=FARBEN_PRISM[k], label=k) for k in KONSTRUKTE]
griffe.append(plt.Line2D([], [], linestyle="none", marker="s", markersize=7,
                         color="#7F7F7F", label="FSK 10 µM (pos ctrl)"))
griffe.append(plt.Line2D([], [], linestyle="--", color="#7F7F7F",
                         label="Fit: log(agonist) vs. response"))
ax.legend(handles=griffe, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

fig.savefig("Abb3_Reporter-gene-assay-Gs.png")
plt.show()

---
## 11. Ergebnisse sichern

Alle Tabellen werden als CSV-Dateien abgelegt (in JupyterLite erscheinen sie im
Dateibrowser links und koennen von dort heruntergeladen werden). Die Abbildungen
wurden bereits als PNG (300 dpi) gespeichert.

In [ ]:
export = {
    "Ergebnis_01_Rohdaten_Triplikate.csv": triplikate,
    "Ergebnis_02_Mittelwerte_SD.csv":      kennwerte,
    "Ergebnis_03_Normiert_x-fold.csv":     ergebnis,
    "Ergebnis_04_Kurvenanpassung.csv":     fits,
}

for name, tabelle in export.items():
    tabelle.to_csv(name, sep=";", decimal=",", encoding="utf-8-sig")
    print("gespeichert:", name)

for name in ["Abb1_Konzentrations-Wirkungs-Kurve.png",
             "Abb2_FSK-Kontrolle.png",
             "Abb3_Reporter-gene-assay-Gs.png"]:
    print("gespeichert:", name)

---
## 12. Qualitaetskontrolle und Interpretation

Bevor die Zahlen biologisch interpretiert werden, muss geprueft werden, ob der Assay
ueberhaupt funktioniert hat. Die folgende Zelle prueft das automatisch anhand von
drei Kriterien:

1. **Signalhoehe** &ndash; liegt die Lumineszenz deutlich ueber dem Geraeterauschen?
2. **Forskolin-Kontrolle** &ndash; FSK umgeht den Rezeptor und muss in *allen* drei
   Konditionen (auch im Leervektor!) ein deutlich erhoehtes Signal erzeugen. Tut es das
   nicht, sind Zellen, Transfektion, Reportergen oder Substrat das Problem &ndash; nicht
   der Rezeptor.
3. **Signalfenster** &ndash; gibt es ueberhaupt einen Unterschied zwischen unstimuliert
   und maximal stimuliert?

In [ ]:
print("=" * 70)
print("QUALITAETSKONTROLLE")
print("=" * 70)

# --- 1) Signalhoehe ------------------------------------------------------
max_signal = float(np.nanmax(kennwerte["Mittelwert"].to_numpy(float)))
print(f"\n1) Signalhoehe")
print(f"   hoechster Mittelwert im Block : {max_signal:,.0f} Counts/s")
print(f"   Referenz (pcDps unstimuliert) : {referenz:,.0f} Counts/s")
if max_signal < 1000:
    print("   ! Alle Werte liegen im Bereich des Geraeterauschens (< 1.000 Counts/s).")
else:
    print("   Signal deutlich ueber dem Rauschen.")

# --- 2) Forskolin-Kontrolle ---------------------------------------------
print(f"\n2) Forskolin-Kontrolle (muss in allen Konditionen >> 1 sein)")
fsk_ok = True
for konstrukt in KONSTRUKTE:
    wert = kennwerte.loc[(konstrukt, "FSK"), "x-fold"]
    basis = kennwerte.loc[(konstrukt, "DMEM"), "x-fold"]
    verh  = wert / basis if basis else np.nan
    status = "ok" if verh >= 2 else "! zu gering"
    fsk_ok &= verh >= 2
    print(f"   {konstrukt:<12} FSK = {wert:6.2f} x-fold   "
          f"(FSK/DMEM = {verh:5.2f})   {status}")

# --- 3) Signalfenster ----------------------------------------------------
print(f"\n3) Signalfenster der KWK (maximal stimuliert / unstimuliert)")
for konstrukt in KONSTRUKTE:
    basis = kennwerte.loc[(konstrukt, "DMEM"), "x-fold"]
    top   = float(np.nanmax(kennwerte.loc[konstrukt].loc[KONZENTRATIONEN, "x-fold"].to_numpy(float)))
    print(f"   {konstrukt:<12} {top / basis:5.2f}-fach" if basis else f"   {konstrukt:<12}   -")

# --- 4) Streuung der Triplikate -----------------------------------------
vk_median = float(np.nanmedian(kennwerte["VK [%]"].to_numpy(float)))
print(f"\n4) Streuung der Triplikate: Median-VK = {vk_median:.1f} %")
if vk_median > 20:
    print("   ! Hohe technische Streuung (VK > 20 %).")

# --- Gesamtbewertung -----------------------------------------------------
print("\n" + "=" * 70)
if max_signal < 1000 or not fsk_ok:
    print("GESAMTBEWERTUNG: Der Assay hat in diesem Block NICHT funktioniert.")
    print("Die EC50-Werte aus Abschnitt 9 sind damit NICHT interpretierbar.")
    print("Moegliche Ursachen: fehlgeschlagene Transfektion, Zellverlust beim")
    print("Mediumwechsel, Substrat- oder Pipettierfehler.")
else:
    print("GESAMTBEWERTUNG: Assay plausibel - EC50-Werte koennen interpretiert werden.")
print("=" * 70)

---
## 13. Zusammenfassung

**Rechenweg**

1. Rohdaten (Counts/s) aus dem Tecan-Export, Block Spalten 13&ndash;18 / Zeilen A&ndash;P
2. Zuordnung ueber das Pipettierschema (Skript S. 28, Abb. 7 und 8)
3. Mittelwert der Triplikate je Konstrukt und Bedingung
4. Normierung: Mittelwert / Mittelwert(pcDps, 0 M) = *x-fold of pcDps unstimulated*
5. Fit `Y = Bottom + (Top - Bottom) / (1 + 10^(LogEC50 - X))` an DMEM (X = &minus;13)
   und 10<sup>&minus;11</sup> &hellip; 10<sup>&minus;6</sup> M &rarr; EC<sub>50</sub>

**Erwartetes Ergebnis (Literatur / Vorlage)**

* MC4R (wt): konzentrationsabhaengiger Signalanstieg, EC<sub>50</sub> im niedrigen nanomolaren Bereich
* mut. MC4R (Ile194Phe): reduziertes Maximalsignal und/oder Rechtsverschiebung der Kurve
  &rarr; Funktionsverlust, passend zum klinischen Bild der monogenen Adipositas
* pcDps: keine Konzentrationsabhaengigkeit (flach bei ca. 1)
* FSK: deutliche Erhoehung in **allen** Konditionen, da Forskolin die Adenylylcyclase
  direkt aktiviert

Ob die eigenen Daten dieses Muster zeigen, beantwortet die Qualitaetskontrolle in
Abschnitt 12.